#  Métodos basados en árboles: árboles de decisión y random forest

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** resistencia a compresión del hormigón (1030 mezclas ensayadas · dataset de Yeh, UCI Machine Learning Repository), como **regresión** (MPa) y como **clasificación** (¿alta resistencia, más de 40 MPa?)

> 🎯 **Mensaje clave:** un árbol son reglas legibles; muchos árboles juntos predicen mucho mejor.

> ✏️ **VERSIÓN ALUMNO.** Rellena los huecos `___` y comenta las preguntas en clase

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Punto de partida: la tabla de modelos de regresión |
| 2 | Un árbol de regresión: leer sus reglas |
| 3 | ¿Hasta dónde dejamos crecer el árbol? |
| 4 | Random forest: muchos árboles mejor que uno |
| 5 | Clasificación: ¿alta resistencia (> 40 MPa)? |

### Imports y configuración básica

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)     # avisos del Lasso con muchas columnas
warnings.filterwarnings("ignore", category=UndefinedMetricWarning) # precisión del modelo «tonto»

# Colores que usaremos en todo el bloque
AZUL = "#4C72B0"      # datos de entrenamiento / clase baja
NARANJA = "#DD8452"   # datos de prueba / validación / clase alta
ROJO = "#C44E52"      # modelo
VERDE = "#55A868"     # árboles
GRIS = "#7F7F7F"

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)


def rmse(y_real, y_pred):
    """Raíz del error cuadrático medio, en las unidades de la respuesta (MPa)."""
    return np.sqrt(mean_squared_error(y_real, y_pred))

---
## 1. Punto de partida: la tabla de modelos de regresión

Mismo fichero, **mismo reparto** (20 % de prueba, que no tocamos) y **mismos 5 pliegues** de validación cruzada que hasta ahora: así los números de la tabla son comparables.

In [ ]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
ruta = "/workspaces/AnaliticaDeDatos/datos/clases/clase_1_analisis_exploratorio_de_datos/Concrete_Data.xls"
concrete_data = pd.read_excel(ruta, index_col=False)

concrete_data.columns = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
                         "arido_grueso", "arido_fino", "edad", "resistencia"]

variables = ["cemento", "escoria", "cenizas", "agua", "superplastificante",
             "arido_grueso", "arido_fino", "edad"]
X = concrete_data[variables]
y = concrete_data["resistencia"]

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)   # mismos pliegues para todo el notebook

# Variables con criterio de ingeniero: relación agua/cemento y log(edad)
X_ing = X.copy()
X_ing["agua_cemento"] = X["agua"] / X["cemento"]
X_ing["log_edad"] = np.log(X["edad"])
X_ing = X_ing.drop(columns="edad")
X_ing_ent, X_ing_pru = X_ing.loc[X_ent.index], X_ing.loc[X_pru.index]   # las mismas mezclas

print("Entrenamiento:", len(X_ent), "mezclas · Prueba:", len(X_pru), "mezclas (no las tocamos)")

In [ ]:
tabla_modelos = []


def evaluar_modelo(nombre, modelo, X, y, comentario=""):
    """Validación cruzada de 5 pliegues (siempre los mismos) y una fila nueva en la tabla de modelos."""
    res = cross_validate(modelo, X, y, cv=cv,
                         scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])
    fila = {"Modelo": nombre,
            "MAE (MPa)": round(-res["test_neg_mean_absolute_error"].mean(), 2),
            "RMSE (MPa)": round(-res["test_neg_root_mean_squared_error"].mean(), 2),
            "R²": round(res["test_r2"].mean(), 3),
            "Comentario": comentario}
    tabla_modelos.append(fila)
    return fila


def rmse_cv(modelo, X, y):
    """RMSE medio de validación cruzada (MPa), sin añadir fila a la tabla."""
    return -cross_val_score(modelo, X, y, cv=cv, scoring="neg_root_mean_squared_error").mean()


def polinomio_lasso_cv():
    """Polinomio de grado 3 con Lasso; alpha elegido por validación cruzada (LassoCV)."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(3, include_bias=False),
                         StandardScaler(), LassoCV(cv=cv, max_iter=50000))

Rehacemos las filas de referencia de la tabla (tarda unos 10 segundos):

In [ ]:
evaluar_modelo("Regresión lineal", LinearRegression(), X_ent, y_ent, "8 variables · rígida")
evaluar_modelo("KNN escalado (K = 3)", make_pipeline(StandardScaler(), KNeighborsRegressor(3)),
               X_ent, y_ent, "K elegido por VC")
evaluar_modelo("Polinomio de grado 3 + Lasso", polinomio_lasso_cv(), X_ent, y_ent,
               ">100 columnas ilegibles · extrapola fatal")
evaluar_modelo("Lineal + a/c + log(edad)", LinearRegression(), X_ing_ent, y_ent,
               "9 coeficientes legibles")

pd.DataFrame(tabla_modelos)

---
## 2. Un árbol de regresión: leer sus reglas

Empezamos con un árbol **muy pequeño** para poder leerlo entero. Cada nodo es una pregunta del tipo «¿variable ≤ umbral?»; si la respuesta es **sí** se baja por la izquierda y si es **no**, por la derecha. Cada hoja predice la **media de la resistencia** de las mezclas de entrenamiento que caen en ella.

✏️ **Rellena el hueco:** un árbol de **profundidad 2** (dos preguntas seguidas como máximo).

In [ ]:
arbol_peq = DecisionTreeRegressor(max_depth=___, random_state=42)   # ✏️
arbol_peq.fit(X_ent, y_ent)

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(arbol_peq, feature_names=variables, filled=True, rounded=True,
          impurity=False, precision=1, fontsize=11, ax=ax)
plt.show()

# Las mismas reglas, en texto
print(export_text(arbol_peq, feature_names=variables, decimals=1))

En cada caja: la pregunta, `samples` (cuántas mezclas llegan) y `value` (la resistencia media de esas mezclas, en MPa). Cuanto más intenso el color, más alta la resistencia.

Usamos el árbol con una mezcla nueva:

In [ ]:
mezcla_nueva = pd.DataFrame({"cemento": [400], "escoria": [100], "cenizas": [0], "agua": [160],
                             "superplastificante": [8], "arido_grueso": [950], "arido_fino": [750],
                             "edad": [28]})
print(f"El árbol predice {arbol_peq.predict(mezcla_nueva)[0]:.1f} MPa")
print(f"Valores distintos que puede predecir este árbol: {arbol_peq.get_n_leaves()}")

> ### 💡 Preguntas
>
> 1) ¿Sobre qué variable hace el árbol la **primera** pregunta? ¿Tiene sentido para un ingeniero?
>
> 2) Sigue a mano el camino de la mezcla nueva por el árbol. ¿En qué hoja cae? ¿Coincide con la predicción?
>
> 3) ¿Cuántos valores distintos de resistencia puede predecir este árbol? ¿Qué forma tendrá la predicción si la dibujamos frente a la edad?

---
## 3. ¿Hasta dónde dejamos crecer el árbol?

La **profundidad máxima** (`max_depth`) es el mando de la complejidad del árbol, como K en KNN o `alpha` en el Lasso. Medimos el error en **entrenamiento** y en **validación cruzada** para varias profundidades (`None` = sin límite: crece hasta que las hojas no se pueden partir más).

✏️ **Rellena el hueco:** la profundidad de cada árbol es la variable del bucle, `prof`.

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, None]
err_ent, err_cv = [], []
for prof in profundidades:
    arbol = DecisionTreeRegressor(max_depth=___, random_state=42)   # ✏️
    arbol.fit(X_ent, y_ent)
    err_ent.append(rmse(y_ent, arbol.predict(X_ent)))      # error con las mezclas que ya ha visto
    err_cv.append(rmse_cv(arbol, X_ent, y_ent))            # error con mezclas que no ha visto

etiquetas = [str(p) if p is not None else "sin\nlímite" for p in profundidades]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(etiquetas, err_ent, marker="o", color=AZUL, linewidth=2, label="Entrenamiento")
ax.plot(etiquetas, err_cv, marker="o", color=NARANJA, linewidth=2, label="Validación cruzada")
ax.set_xlabel("Profundidad máxima del árbol"); ax.set_ylabel("RMSE (MPa)")
ax.legend(frameon=False); estilo(ax)
plt.tight_layout(); plt.show()

mejor_prof = profundidades[int(np.argmin(err_cv))]
print(f"Profundidad con menor error de validación: {mejor_prof} → {min(err_cv):.2f} MPa")
pd.DataFrame({"Profundidad": profundidades, "RMSE entrenamiento": np.round(err_ent, 2),
              "RMSE validación": np.round(err_cv, 2)})

> ### 💡 Preguntas
>
> 1) ¿Qué le pasa al error de **entrenamiento** al dejar crecer el árbol? ¿Y al de **validación**?
>
> 2) ¿Dónde hay infraajuste y dónde sobreajuste? ¿Por qué no nos fiamos del error de entrenamiento?

Añadimos dos árboles a la tabla: el que crece sin límite y el de la profundidad elegida.

In [ ]:
evaluar_modelo("Árbol sin límite", DecisionTreeRegressor(random_state=42), X_ent, y_ent,
               "memoriza el entrenamiento")
evaluar_modelo(f"Árbol (profundidad {mejor_prof})", DecisionTreeRegressor(max_depth=mejor_prof, random_state=42),
               X_ent, y_ent, "profundidad elegida por VC")
pd.DataFrame(tabla_modelos)

> ℹ️ Hemos elegido la profundidad con los mismos pliegues con los que la evaluamos, así que su número es ligeramente optimista. Para comparar modelos en clase nos vale.

> ### 💡 Preguntas
>
> 1) ¿Dónde queda el árbol en la tabla? ¿Mejora a la regresión lineal? ¿Y al polinomio con Lasso?
>
> 2) ¿Sigue siendo legible un árbol de esa profundidad? (Pista: `DecisionTreeRegressor(max_depth=mejor_prof).fit(X_ent, y_ent).get_n_leaves()`)

---
## 4. Random forest: muchos árboles mejor que uno

- **Bagging:** cada árbol se entrena con una muestra *bootstrap* (sorteo con reemplazo de las mezclas) y se promedian las predicciones de todos.
- **Random forest:** igual, pero en cada corte el árbol solo puede elegir entre un **sorteo de variables** (`max_features=0.5`: la mitad, 4 de las 8). Así los árboles son más distintos entre sí y el promedio funciona mejor.

En scikit-learn los dos se hacen con `RandomForestRegressor`: con `max_features=None` (todas las variables) es bagging. `n_jobs=-1` usa todos los procesadores del ordenador.

✏️ **Rellena el hueco:** **300** árboles en el random forest (`n_estimators`).

In [ ]:
evaluar_modelo("Bagging (300 árboles)",
               RandomForestRegressor(n_estimators=300, max_features=None, random_state=42, n_jobs=-1),
               X_ent, y_ent, "árboles grandes promediados")
evaluar_modelo("Random forest (300 árboles)",
               RandomForestRegressor(n_estimators=___, max_features=0.5, random_state=42, n_jobs=-1),   # ✏️
               X_ent, y_ent, "además, sorteo de variables")
pd.DataFrame(tabla_modelos)

> ### 💡 Preguntas
>
> 1) ¿Cuánto baja el error al pasar de un árbol a un bosque? ¿Dónde queda ahora el polinomio con Lasso?
>
> 2) ¿Hay mucha diferencia entre bagging y random forest? ¿Por qué creéis que es así con estos datos?

### 4.1. ¿Cuántos árboles hacen falta?

Más árboles **no** provocan sobreajuste: el error baja y se estabiliza. Solo cuesta más tiempo de cálculo.

In [ ]:
n_arboles = [1, 5, 10, 25, 50, 100, 300]
err_bosque = [rmse_cv(RandomForestRegressor(n_estimators=n, max_features=0.5, random_state=42, n_jobs=-1),
                      X_ent, y_ent) for n in n_arboles]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_arboles, err_bosque, marker="o", color=VERDE, linewidth=2)
ax.axhline(min(err_cv), color=GRIS, linestyle="--", label="Mejor árbol individual")
ax.set_xscale("log"); ax.set_xlabel("Número de árboles en el bosque"); ax.set_ylabel("RMSE de validación (MPa)")
ax.legend(frameon=False); estilo(ax); plt.tight_layout(); plt.show()

### 4.2. ¿Y con las variables de ingeniero?

Probamos el bosque con a/c y log(edad), igual que hicimos con la regresión lineal.

In [ ]:
evaluar_modelo("Random forest + a/c + log(edad)",
               RandomForestRegressor(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1),
               X_ing_ent, y_ent, "9 variables con sentido físico")
pd.DataFrame(tabla_modelos).tail(3)

> ### 💡 Preguntas
>
> 1) En la regresión lineal, a/c y log(edad) bajaban el error de 10,6 a 7,2 MPa. ¿Y en el bosque?
>
> 2) ¿Por qué el logaritmo de la edad no le aporta nada a un árbol?

### 4.3. Importancia de variables

Con 300 árboles ya no se pueden leer las reglas. A cambio, el bosque nos dice **cuánto reduce el error cada variable**, sumado en todos los cortes de todos los árboles (las importancias suman 1).

✏️ **Rellena el hueco:** las importancias están en el atributo `feature_importances_` del bosque entrenado.

In [ ]:
bosque = RandomForestRegressor(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
bosque.fit(X_ing_ent, y_ent)

importancia = pd.Series(bosque.___, index=X_ing.columns).sort_values()   # ✏️

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importancia.index, importancia.values, color=VERDE)
ax.set_xlabel("Importancia (reducción del error, suma = 1)"); estilo(ax)
plt.tight_layout(); plt.show()

importancia.sort_values(ascending=False).round(3).to_frame("Importancia")

> ### 💡 Preguntas
>
> 1) ¿Qué variables usa más el bosque? ¿Acertasteis la apuesta?
>
> 2) Comparad con los coeficientes de la lineal con a/c y log(edad), donde a/c era negativo y log(edad) positivo. ¿Coinciden las variables? ¿Qué información nos da la lineal que la importancia no da?

### 4.4. Comprobación final con las mezclas de prueba

Comparamos **una sola vez** sobre las 206 mezclas de prueba los dos candidatos: el modelo legible (lineal con a/c y log(edad)) y el más preciso (random forest).

✏️ **Rellena el hueco:** las predicciones se obtienen con `predict`.

In [ ]:
lineal_ing = LinearRegression().fit(X_ing_ent, y_ent)
pred_lineal = lineal_ing.predict(X_ing_pru)
pred_bosque = bosque.___(X_ing_pru)   # ✏️

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, pred, nombre, color in [(axes[0], pred_lineal, "Lineal + a/c + log(edad)", AZUL),
                                (axes[1], pred_bosque, "Random forest + a/c + log(edad)", VERDE)]:
    ax.scatter(y_pru, pred, s=18, alpha=0.6, color=color)
    ax.plot([0, 85], [0, 85], color=ROJO, linewidth=1.5)
    ax.set_title(f"{nombre}\nRMSE en prueba = {rmse(y_pru, pred):.1f} MPa", fontsize=12)
    ax.set_xlabel("Resistencia observada (MPa)"); estilo(ax)
axes[0].set_ylabel("Resistencia predicha (MPa)")
plt.tight_layout(); plt.show()

### La tabla de modelos de regresión, completa

In [ ]:
tabla = pd.DataFrame(tabla_modelos).sort_values("RMSE (MPa)").reset_index(drop=True)
display(tabla)

fig, ax = plt.subplots(figsize=(9, 5))
colores = [VERDE if ("Árbol" in m or "forest" in m or "Bagging" in m) else AZUL for m in tabla["Modelo"]]
ax.barh(tabla["Modelo"], tabla["RMSE (MPa)"], color=colores)
ax.invert_yaxis(); ax.set_xlabel("RMSE de validación cruzada (MPa)")
for i, v in enumerate(tabla["RMSE (MPa)"]):
    ax.text(v + 0.1, i, f"{v:.1f}", va="center", fontsize=10)
estilo(ax); plt.tight_layout(); plt.show()

tabla.to_csv("tabla_modelos_hormigon.csv", index=False)
print("Tabla guardada en tabla_modelos_hormigon.csv")

> ### 💡 Pregunta
>
> 1) ¿Qué modelo llevaríais ahora a la planta? ¿Cambia vuestra respuesta respecto a la del Lasso?

> 🔎 **Opcional (boosting):** añade una fila con `GradientBoostingRegressor` (en `sklearn.ensemble`): árboles pequeños en secuencia, cada uno corrigiendo los errores de los anteriores.
>
> ```python
> from sklearn.ensemble import GradientBoostingRegressor
> evaluar_modelo("Boosting", GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42), X_ent, y_ent, "árboles en secuencia")
> ```

---
## 5. Clasificación: ¿alta resistencia (> 40 MPa)?

Misma definición que con la regresión logística: **clase 1 = alta resistencia** (más de 40 MPa) y **clase 0 = baja**. Los árboles sirven igual: cada hoja predice la **clase mayoritaria** de sus mezclas y la proporción de altas en la hoja es la **probabilidad**.

Métricas (las mismas que con la logística):

- **Exactitud**: porcentaje de mezclas bien clasificadas.
- **Sensibilidad** (*recall*): de las que **son** altas, cuántas detecta el modelo.
- **Precisión**: de las que el modelo **dice** que son altas, cuántas lo son de verdad. Un «alta» equivocado es una mezcla que se da por buena sin llegar.

In [ ]:
umbral = 40
y_ent_log = (y_ent > umbral).astype(int)
y_pru_log = (y_pru > umbral).astype(int)
print(f"Mezclas de alta resistencia en entrenamiento: {y_ent_log.mean():.0%}")

tabla_clasificadores = []


def evaluar_clasificador(nombre, modelo, X, y, comentario=""):
    """Validación cruzada de 5 pliegues (siempre los mismos) y una fila nueva en la tabla de clasificadores."""
    res = cross_validate(modelo, X, y, cv=cv, scoring=["accuracy", "recall", "precision"])
    fila = {"Modelo": nombre,
            "Exactitud": round(res["test_accuracy"].mean(), 3),
            "Sensibilidad": round(res["test_recall"].mean(), 3),
            "Precisión": round(res["test_precision"].mean(), 3),
            "Comentario": comentario}
    tabla_clasificadores.append(fila)
    return fila


def logistica_escalada():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))


evaluar_clasificador("Siempre la clase más frecuente", DummyClassifier(strategy="most_frequent"),
                     X_ent, y_ent_log, "referencia mínima")
evaluar_clasificador("Logística · 8 variables", logistica_escalada(), X_ent, y_ent_log, "edad en días")
evaluar_clasificador("Logística · a/c + log(edad)", logistica_escalada(), X_ing_ent, y_ent_log,
                     "9 variables con sentido físico")
pd.DataFrame(tabla_clasificadores)

### 5.1. Un árbol clasificador pequeño

✏️ **Rellena el hueco:** para clasificar se usa `DecisionTreeClassifier` (en lugar de `DecisionTreeRegressor`).

In [ ]:
arbol_clas = ___(max_depth=3, random_state=42)   # ✏️
arbol_clas.fit(X_ent, y_ent_log)

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(arbol_clas, feature_names=variables, class_names=["Baja", "Alta"], filled=True, rounded=True,
          impurity=False, proportion=True, precision=2, fontsize=10, ax=ax)
plt.show()

En cada caja, `value` es ahora la **proporción** de mezclas bajas y altas que llegan a ese nodo, y `class` la clase mayoritaria (naranja = baja, azul = alta; cuanto más intenso, más pura).

> ### 💡 Preguntas
>
> 1) Elegid una hoja azul: ¿qué reglas debe cumplir una mezcla para llegar a ella? ¿Con qué probabilidad es alta?
>
> 2) ¿Se parecen las preguntas a las del árbol de regresión?

### 5.2. La tabla de clasificadores

Elegimos la profundidad del árbol por validación cruzada (ahora con la **exactitud**) y comparamos con el random forest.

✏️ **Rellena el hueco:** el bosque para clasificar es `RandomForestClassifier`.

In [ ]:
exactitud_cv = [cross_val_score(DecisionTreeClassifier(max_depth=p, random_state=42), X_ent, y_ent_log,
                                cv=cv, scoring="accuracy").mean() for p in profundidades]
mejor_prof_clas = profundidades[int(np.argmax(exactitud_cv))]
print(f"Profundidad con mayor exactitud de validación: {mejor_prof_clas}")

evaluar_clasificador("Árbol (profundidad 3)", DecisionTreeClassifier(max_depth=3, random_state=42),
                     X_ent, y_ent_log, "legible")
evaluar_clasificador(f"Árbol (profundidad {mejor_prof_clas})",
                     DecisionTreeClassifier(max_depth=mejor_prof_clas, random_state=42),
                     X_ent, y_ent_log, "profundidad elegida por VC")
evaluar_clasificador("Random forest (300 árboles)",
                     ___(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1),   # ✏️
                     X_ent, y_ent_log, "8 variables")
evaluar_clasificador("Random forest + a/c + log(edad)",
                     RandomForestClassifier(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1),
                     X_ing_ent, y_ent_log, "9 variables con sentido físico")
pd.DataFrame(tabla_clasificadores)

> ### 💡 Preguntas
>
> 1) ¿Qué clasificador gana? ¿En qué métricas?
>
> 2) El árbol de profundidad 3, ¿se queda muy por detrás de la logística? ¿Qué ofrece a cambio?

### 5.3. Matriz de confusión con las mezclas de prueba

Comparamos **una sola vez**, con las mezclas de prueba, la mejor logística y el random forest con las mismas variables.

|  | Predice baja | Predice alta |
|---|:--:|:--:|
| **Es baja** | ✅ verdadero negativo | ⚠️ **falso positivo**: se acepta una mezcla que no llega |
| **Es alta** | falso negativo: se rechaza una mezcla buena | ✅ verdadero positivo |

✏️ **Rellena el hueco:** las predicciones sobre las mezclas de prueba se obtienen con `predict`.

In [ ]:
logistica_ing = logistica_escalada().fit(X_ing_ent, y_ent_log)
bosque_clas = RandomForestClassifier(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1).fit(X_ing_ent, y_ent_log)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, modelo, nombre in [(axes[0], logistica_ing, "Logística · a/c + log(edad)"),
                           (axes[1], bosque_clas, "Random forest · a/c + log(edad)")]:
    pred = modelo.___(X_ing_pru)   # ✏️
    ConfusionMatrixDisplay(confusion_matrix(y_pru_log, pred), display_labels=["Baja", "Alta"]).plot(
        ax=ax, cmap="Blues", colorbar=False)
    vn, fp, fn, vp = confusion_matrix(y_pru_log, pred).ravel()
    ax.set_title(f"{nombre}\nExactitud {(vn + vp) / len(pred):.1%} · FP = {fp} · FN = {fn}", fontsize=11)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Realidad")
plt.tight_layout(); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Cuántas mezclas que **no** llegan a 40 MPa da por buenas cada modelo (falsos positivos)? ¿Cuál preferiría la planta?
>
> 2) Si aun así el número de falsos positivos del bosque os parece alto, ¿qué podríais mover, como se hizo con la logística?

In [ ]:
# Opcional: mover el umbral de decisión del bosque
prob_alta = bosque_clas.predict_proba(X_ing_pru)[:, 1]
for u in [0.5, 0.6, 0.7, 0.8]:
    pred_u = (prob_alta >= u).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_pru_log, pred_u).ravel()
    print(f"Umbral {u:.1f} → falsos positivos: {fp:3d} · falsos negativos: {fn:3d}")

### La tabla de clasificadores, completa

In [ ]:
tabla_clas = pd.DataFrame(tabla_clasificadores).sort_values("Exactitud", ascending=False).reset_index(drop=True)
display(tabla_clas)
tabla_clas.to_csv("tabla_clasificadores_hormigon.csv", index=False)
print("Tabla guardada en tabla_clasificadores_hormigon.csv")

---
### 📌 Para llevarse

- **Un árbol son reglas «si… entonces…»:** parte los datos con preguntas sobre una variable y un umbral. En regresión, la hoja da la media (MPa); en clasificación, la clase mayoritaria y su proporción.
- **Sin freno, un árbol memoriza:** la profundidad es el mando de la complejidad y se elige por validación cruzada. No hace falta escalar las variables.
- **Muchos árboles distintos predicen mucho mejor:** bagging y random forest promedian árboles entrenados con muestras y variables sorteadas. En el hormigón, el bosque es el mejor modelo de las dos tablas.
- **Se gana precisión, se pierde lectura:** la importancia de variables resume qué usa el bosque (edad y a/c, como diría un ingeniero), pero sin signo ni reglas.